In [1]:
from fastapi import FastAPI
from copilotkit.integrations.fastapi import add_fastapi_endpoint
from copilotkit import CopilotKitSDK, Action as CopilotAction
import google.generativeai as genai
from openai import OpenAI
import weave
from pydantic import BaseModel
import typing_extensions as typing
import os
import logging
import json

/Users/kaitlynhennacy/.local/share/virtualenvs/ghost_agent_hackathon-a8AUSSkz/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
async def create_written_story(notes: str):
    genai.configure(api_key=os.environ["GOOGLE_GEMINI_API_KEY"])
    model = genai.GenerativeModel(model_name="gemini-1.5-pro", system_instruction="You are a helpful assistant that creates a well-written story based on the provided notes and outline. You return just the story, no other text.")
    prompt = f"Create a well-written story based on the following notes and outline in JSON format: {notes}\n\nYour story will be in raw text and 500 words."
    response = model.generate_content(prompt)

    return response.text

class StoryRequest(BaseModel):
    notes: str

class StoryOutline(BaseModel):
    title: str
    themes: list[str]
    characters: list[str]
    setting: str
    plot_points: list[str]
    writing_style_instructions: str

class StoryOutlineRequest(BaseModel):
    conversation: str
    previous_outline: str

class ReplyRequest(BaseModel):
    previous_message: str
    current_outline: str
    current_story: str

class RouterResponse(BaseModel):
    reply_with: str





In [5]:
conversation_history = []
while True:
    current_outline = """
    Themes: empty
    Characters: empty
    Setting: empty
    Plot points: empty
    Writing style instructions: empty
    """
    user_message = input("User: ")
    if not user_message:
        break
    conversation_history.append("User: " +  user_message)
    def reply_to_user(request: ReplyRequest):
        #client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])   
        completion = client.beta.chat.completions.parse(
                model="gpt-4o", 
                messages=[
                    {"role": "system", "content": """
                    You will determine whether the user is asking you to output a written draft now. If the user's intent is to get a written draft immediately, then
                        you will output "WRITE". Otherwise, you will output "CHAT".
                    """},
                    {"role": "user", "content": f"""{request.previous_message}
                    """}
                ],
                response_format=RouterResponse
            )
        reply_with = completion.choices[0].message.parsed.reply_with

        if reply_with == "CHAT":
             system_prompt = """You are a ghostwriter interviewing a person to write their story. 
    You need to reply to the user's message in a way that helps you enrich their story. Ask good engaging questions like an experienced ghostwriter would. Only ask one question at a time."""
        else:
             system_prompt = """You are a ghostwriter writing a story based on the provided outline. You will output a written draft of the story."""
        genai.configure(api_key=os.environ["GOOGLE_GEMINI_API_KEY"])
        model = genai.GenerativeModel(
            model_name="gemini-1.5-flash", system_instruction=system_prompt
        )
        prompt = f"""This is the story outine you have so far:\n{request.current_outline}\n\n
        This is what you have written for the story so far:\n\n {request.current_story}
        Message from the user:\n\n {request.previous_message}
        """
        response = model.generate_content(prompt)
        return response.text
    #     completion = client.chat.completions.create(
    #         model="gpt-4o-mini", 
    #         messages=[
    #             {"role": "system", "content": """You are a ghostwriter interviewing a person to write their story. 
    # If asked to output the completed story, and the outline is complete, you will write the story as requested based on the outline.
    # Otherwise, you need to reply to the user's message in a way that helps you enrich their story. Ask good engaging questions like a ghostwriter would. Only ask one question at a time."""},
    #             {"role": "user", "content": f"""This is the story outine you have so far:\n{request.current_outline}\n\n
    #             This is what you have written for the story so far:\n\n {request.current_story}
    #             Reply to this message:\n\n {request.previous_message}
    #             """}
    #         ]
    #     )
    #     return completion.choices[0].message.content

    def create_story_outline(request: StoryOutlineRequest):
        client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])   
        completion = client.beta.chat.completions.parse(
            model="gpt-4o", 
            messages=[
                {"role": "system", "content": """
                 You are AI ghostwriter that is filling out a story outline based on the provided notes. 
                 ONLY extract information fromthe words of the client. If you have no content for a field, you MUST fill in with the word 'EMPTY'.
                 Then think about whether you made up the field or not. If it is made up, you MUST replace it with the word 'EMPTY'.
                 Again, NOTHING in the outline can be made up and it is VERY likely that some fields will be EMPTY.
                 """},
                {"role": "user", "content": f"""This is the previous outline:\n{request.previous_outline}\n\n
                Create a new story outline.
                
                Return a JSON object with these exact fields:
                - themes: string
                - characters: string
                - setting: string
                - plot_points: string
                - writing_style_instructions: string
                
                Incorporate the newest information from the following conversation:\n\n {request.conversation}
                """}
            ],
            response_format=StoryOutline
        )
        return f"""
        Themes: {completion.choices[0].message.parsed.themes}
        Characters: {completion.choices[0].message.parsed.characters}
        Setting: {completion.choices[0].message.parsed.setting}
        Plot points: {completion.choices[0].message.parsed.plot_points}
        Writing style instructions: {completion.choices[0].message.parsed.writing_style_instructions}
        """

    reply = reply_to_user(ReplyRequest(previous_message=user_message, current_outline=current_outline, current_story=""))
    print(reply)
    print("---------------Outline-----------------")
    current_outline = create_story_outline(StoryOutlineRequest(previous_outline=current_outline, conversation="\n".join(conversation_history)))
    print(current_outline)
    conversation_history.append("Ghostwriter: " + reply)
    print("--------------------------------")
    


That's a great start!  Henry the frog sounds like a compelling character.  What kind of frog is Henry?  Is he a common garden variety, or something more exotic? 

---------------Outline-----------------

        Themes: ['EMPTY']
        Characters: ['a frog named Henry']
        Setting: EMPTY
        Plot points: ['EMPTY']
        Writing style instructions: EMPTY
        
--------------------------------


In [19]:
fake_conversations = [
    ["Can you write me the story?", """
Characters: Kaitlyn 
Themes: Learning that life is short and you should live it to the fullest.
Setting: New York City. Plot points: Kaitlyn learns that life is short and she should live it to the fullest
Plot points: She goes to the Met, eats bagels, and bikes around the city.
Writing Style Instructions: Write it in the style of memoir by stephanie land"""],
]

for fake_conversation in fake_conversations:
    print(reply_to_user(ReplyRequest(previous_message=fake_conversation[0], current_outline=fake_conversation[1])))

ValidationError: 1 validation error for ReplyRequest
current_story
  Field required [type=missing, input_value={'previous_message': 'Can...moir by stephanie land'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.9/v/missing